# 01. 할인 패턴과 다음 할인 예측 (A1)

> 겜누리 · 분석 A1 · 데이터: 수집한 PC 스토어 가격 이력(`price_history`, MariaDB Replica)

## 1단계. 데이터 분석 설계

### 왜 이 분석을 하나
서비스의 "다음 할인 예상일"은 **마지막 할인 시작일 + 과거 할인 간격의 평균**이라는 단순 규칙으로 계산한다
(`backend/app/services/timing.py`). 이 규칙이 통계적으로 타당한지, 더 나은 기준은 없는지 확인한다.

### 가설
| # | 가설 | 귀무가설 (H0) | 검정 | 왜 이 방법인가 |
|---|---|---|---|---|
| H1 | 할인은 **무작위가 아니라 규칙적인 간격**으로 반복된다 | 간격이 무작위(포아송 과정)다 | 간격 변동계수(CV)를 시뮬레이션 기준값과 비교, 대응 Wilcoxon 부호순위 검정 | 간격이 규칙적이면 CV가 작다. 무작위(지수분포)면 CV≈1이고, 표본이 적으면 CV가 편향되므로 게임별 시뮬레이션 기준값과 비교한다 |
| H2 | 할인은 **특정 달에 몰린다** (스토어 세일 시즌) | 12개월에 고르게 퍼진다 | 카이제곱 적합도 검정, Cramér's V | 월별 "횟수"를 비교하는 범주형 검정. 관찰 기간이 달마다 다르므로 기대값을 **관찰 일수 비율**로 둔다 |
| H3 | 서비스의 **평균 간격 규칙이 단순한 기준보다 정확하다** | 규칙의 예측 오차가 기준과 같다 | 대응 Wilcoxon 부호순위 검정 (오차 비교) | 오차가 정규분포가 아니고 한쪽으로 치우쳐서 비모수 검정을 쓴다 |

### 분석 도구
`pandas`(정제·집계) · `scipy.stats`(검정) · `matplotlib`(시각화) · (선택) `prophet`(계절성 확인)

**데이터 한계**: ITAD 응답이 최근 약 23개월이라 연간 패턴은 최대 2번까지만 본다. 분석 대상 1,000개는 한국어 리뷰가 많은 인기작 위주라 전체 게임으로 일반화하기는 어렵다.

## 2단계. 데이터 준비

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import platform
from functools import lru_cache
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    plt.rc("font", family="AppleGothic")
plt.rc("axes", unicode_minus=False)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# 레포 루트 찾기 (노트북을 어디서 열어도 같은 경로를 쓴다)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "analysis").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "analysis" / "data"
print("레포 루트:", ROOT)

In [ ]:
# 가격 이력은 DB(Replica)에서 SELECT로만 읽고, 한 번 읽으면 CSV로 저장해 다음부터 재사용한다.
# (발표 현장에서 VM·SSH 터널이 안 돼도 시연할 수 있게 하기 위함. analysis/data/ 는 Git에 올라가지 않는다.)
PRICE_CSV = Path(os.environ.get("GAMENURI_PRICE_CSV", DATA / "cache" / "price_history_pc_new.csv"))


def load_price_history(game_ids, refresh: bool = False) -> pd.DataFrame:
    """PC·새 상품(item_condition='new') 가격 이력. 컬럼: game_id, store, edition, price_krw, discount_pct, collected_at"""
    if PRICE_CSV.exists() and not refresh:
        df = pd.read_csv(PRICE_CSV, parse_dates=["collected_at"])
    else:
        from sqlalchemy import bindparam, create_engine, text
        from gamenuri_common.config import Settings

        s = Settings(_env_file=ROOT / ".env")  # 비밀번호는 출력하지 않는다
        url = (f"mysql+pymysql://{s.mariadb_user}:{quote_plus(s.mariadb_password)}"
               f"@{s.mariadb_read_host}:{s.mariadb_port}/{s.mariadb_db}")
        query = text(
            "SELECT game_id, store, edition, price_krw, regular_krw, discount_pct, collected_at "
            "FROM price_history WHERE platform = 'PC' AND item_condition = 'new' AND game_id IN :ids"
        ).bindparams(bindparam("ids", expanding=True))
        with create_engine(url).connect() as conn:  # 읽기 전용 계정, SELECT만 실행
            df = pd.read_sql(query, conn, params={"ids": [int(i) for i in game_ids]})
        PRICE_CSV.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(PRICE_CSV, index=False)
    return df[df.game_id.isin(game_ids)].copy()

In [ ]:
games = pd.read_csv(ROOT / "analysis" / "game_list" / "games_1000.csv", encoding="utf-8-sig", parse_dates=["release_date"])
games["name_ko"] = games.name_ko.fillna(games.name)   # 한글 이름이 없으면 영문 이름
prices = load_price_history(games.game_id.tolist())

print(f"행 {len(prices):,} · 게임 {prices.game_id.nunique():,}개 · "
      f"기간 {prices.collected_at.min():%Y-%m-%d} ~ {prices.collected_at.max():%Y-%m-%d}")
prices.head()

In [ ]:
# 구조 파악: 컬럼 타입과 결측치
prices.info()
prices.isna().sum()

In [ ]:
# 게임별 이력이 몇 일치인지 (H1~H3은 이력이 어느 정도 있어야 의미가 있다)
hist_days = prices.assign(day=prices.collected_at.dt.normalize()).groupby("game_id").day.nunique()
print(hist_days.describe().round(1))
hist_days.plot.hist(bins=40, figsize=(7, 3.5), title="게임별 가격 이력 일수")
plt.xlabel("일수"); plt.show()

### 차트 해설 — 게임별 이력 일수
이력이 매우 짧은 게임(몇 건뿐)은 간격을 구할 수 없다. 이후 단계에서 **할인이 4회 이상 있는 게임**만 간격 분석에 쓴다.

## 3단계. 데이터 가공

- 스토어별 마지막 가격을 이어 채워(forward fill) 날짜별 **최저가 한 줄**로 만든다.
- 할인율 10% 이상인 날이 이어진 구간을 **할인 1회**로 센다. 서비스(`timing.py`)와 같은 상수를 써서 "서비스 기준"으로 분석한다.

In [ ]:
# ---- 서비스(backend/app/services/timing.py)와 같은 기준 ----
SALE_MIN_PCT = 10      # 할인율 10% 이상인 날이 이어진 구간 = 할인 1회
WINDOW_DAYS = 730      # 할인 횟수·주기 계산 기간 (최근 2년)
MIN_HISTORY_DAYS = 3   # 이력 일수가 이보다 적으면 제외


def daily_series(g: pd.DataFrame) -> pd.DataFrame:
    """게임 1개의 일별 최저가. 스토어(+에디션)별 마지막 가격을 이어 채운 뒤 그날의 최저가 한 줄만 남긴다.
    (price_history에는 가격이 바뀐 스토어의 행만 있어서 이렇게 해야 모든 스토어의 '그 시점 가격'이 된다)"""
    g = g.assign(day=g.collected_at.dt.normalize(), discount_pct=g.discount_pct.fillna(0))
    last = g.sort_values("collected_at").groupby(["day", "store", "edition"]).tail(1)
    price = last.pivot(index="day", columns=["store", "edition"], values="price_krw")
    disc = last.pivot(index="day", columns=["store", "edition"], values="discount_pct")
    price = price.sort_index().sort_index(axis=1).ffill()
    disc = disc.reindex_like(price).ffill()
    best = np.nanargmin(price.to_numpy(), axis=1)
    rows = np.arange(len(best))
    return pd.DataFrame({
        "day": price.index,
        "price_krw": price.to_numpy()[rows, best],
        "discount_pct": disc.to_numpy()[rows, best],
    })


def sale_events(d: pd.DataFrame) -> pd.DataFrame:
    """할인율이 SALE_MIN_PCT 이상인 날이 이어진 구간마다 한 건 (시작일, 최대 할인율)."""
    events, start, top = [], None, 0
    for day, pct in zip(d.day, d.discount_pct):
        if pct >= SALE_MIN_PCT:
            if start is None:
                start, top = day, pct
            else:
                top = max(top, pct)
        elif start is not None:
            events.append((start, top))
            start = None
    if start is not None:
        events.append((start, top))
    return pd.DataFrame(events, columns=["start", "max_pct"]).astype({"start": "datetime64[ns]"})

In [ ]:
END = prices.collected_at.max().normalize()   # 데이터의 마지막 날을 "지금"으로 본다
SINCE = END - pd.Timedelta(days=WINDOW_DAYS)

series_by_game = {gid: daily_series(g) for gid, g in prices.groupby("game_id")}
series_by_game = {gid: d for gid, d in series_by_game.items() if len(d) >= MIN_HISTORY_DAYS}
events = {gid: sale_events(d[d.day >= SINCE]) for gid, d in series_by_game.items()}

summary = pd.DataFrame({
    "n_events": {gid: len(ev) for gid, ev in events.items()},
}).join(games.set_index("game_id")[["name_ko", "genres"]])
print(f"분석 가능 게임 {len(summary):,}개 · 할인 횟수 중앙값 {summary.n_events.median():.0f}회")
summary.n_events.plot.hist(bins=range(0, int(summary.n_events.max()) + 2), figsize=(7, 3.5),
                           title="게임별 할인 횟수 (최근 2년)")
plt.xlabel("할인 횟수"); plt.show()

### 차트 해설 — 게임별 할인 횟수
할인이 많은 게임과 거의 없는 게임이 섞여 있다. 간격을 구하려면 최소 2회, 신뢰할 만한 변동계수는 **4회 이상**(간격 3개 이상)이 필요하다.

In [ ]:
MIN_EVENTS = 4  # 간격이 3개 이상 나오는 게임만 H1·H3에 사용

gap_rows = []
for gid, ev in events.items():
    if len(ev) >= MIN_EVENTS:
        gaps = ev.start.diff().dt.days.dropna().to_numpy()
        gap_rows.append({"game_id": gid, "n_events": len(ev), "gaps": gaps,
                         "mean_gap": gaps.mean(), "cv": gaps.std(ddof=1) / gaps.mean()})
gap_df = pd.DataFrame(gap_rows).set_index("game_id")
print(f"간격 분석 대상 {len(gap_df):,}개 게임 (할인 {MIN_EVENTS}회 이상)")
gap_df[["n_events", "mean_gap", "cv"]].describe().round(2)

## 4단계. 데이터 분석

### H1. 할인 간격은 규칙적인가?
- 간격의 **변동계수(CV = 표준편차/평균)** 가 작을수록 규칙적이다.
- 할인이 완전히 무작위(지수분포 간격)이면 CV는 약 1이다. 다만 간격 수가 적으면 표본 CV가 1보다 작게 나오는 편향이 있어서,
  **같은 간격 수를 가진 무작위 시뮬레이션의 평균 CV**를 게임마다 기준값으로 둔다.

In [ ]:
rng = np.random.default_rng(42)

@lru_cache(maxsize=None)
def null_cv(k: int, n_sim: int = 5000) -> float:
    """간격이 k개인 무작위(지수분포) 과정의 평균 CV."""
    x = rng.exponential(1.0, size=(n_sim, k))
    return float((x.std(axis=1, ddof=1) / x.mean(axis=1)).mean())

gap_df["cv_null"] = [null_cv(len(g)) for g in gap_df.gaps]
w1 = stats.wilcoxon(gap_df.cv, gap_df.cv_null, alternative="less")
d1 = gap_df.cv - gap_df.cv_null
print(f"관측 CV 중앙값 {gap_df.cv.median():.2f}  vs  무작위 기준 CV 중앙값 {gap_df.cv_null.median():.2f}")
print(f"Wilcoxon 부호순위 검정(관측 < 무작위): 통계량 {w1.statistic:.0f}, p = {w1.pvalue:.4g}")
print(f"무작위보다 규칙적인 게임 비율: {(d1 < 0).mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
pooled = np.concatenate(gap_df.gaps.to_list())
ax[0].hist(pooled, bins=np.arange(0, 400, 10), color="#4c78a8")
ax[0].axvline(np.median(pooled), color="red", ls="--", label=f"중앙값 {np.median(pooled):.0f}일")
ax[0].set(title="할인 시작일 간격 분포", xlabel="일", ylabel="횟수"); ax[0].legend()
ax[1].scatter(gap_df.cv_null, gap_df.cv, s=12, alpha=.5)
lim = [0, max(gap_df.cv.max(), gap_df.cv_null.max()) * 1.05]
ax[1].plot(lim, lim, color="gray", ls="--"); ax[1].set(xlim=lim, ylim=lim)
ax[1].set(title="게임별 CV: 관측 vs 무작위 기준", xlabel="무작위 기준 CV", ylabel="관측 CV")
plt.tight_layout(); plt.show()

### 차트 해설 — H1
- **왼쪽**: 할인 간격이 어느 구간에 몰리는지 본다. 한 구간에 뾰족하게 모이면 주기성이 있다는 뜻이다.
- **오른쪽**: 점이 대각선 **아래**에 있으면 그 게임은 무작위보다 규칙적이다. 대각선 위에 많으면 오히려 불규칙(몰아서 할인하고 길게 쉬는 패턴)이다.

### H2. 할인은 특정 달에 몰리는가?
월별 **할인 시작 횟수**가 균등한지 카이제곱 적합도 검정을 한다. 데이터 기간이 달마다 다르므로(예: 23개월이면 어떤 달은 2번, 어떤 달은 1번 관찰),
기대 횟수는 **그 달을 관찰한 일수 비율**로 둔다.

In [ ]:
exposure = np.zeros(12)
for gid, d in series_by_game.items():
    span = pd.date_range(max(d.day.min(), SINCE), END)
    exposure += np.bincount(span.month - 1, minlength=12)

starts = pd.concat([ev.start for ev in events.values()])
observed = np.bincount(starts.dt.month - 1, minlength=12)
expected = exposure / exposure.sum() * observed.sum()
chi2, p_month = stats.chisquare(observed, expected)
cramers_v = np.sqrt(chi2 / (observed.sum() * 11))
print(f"할인 시작 {observed.sum():,}건 · 카이제곱 {chi2:.1f} (자유도 11), p = {p_month:.4g}, Cramér's V = {cramers_v:.2f}")

ratio = pd.Series(observed / expected, index=range(1, 13))
ax = ratio.plot.bar(figsize=(9, 3.8), color=np.where(ratio > 1.15, "#e45756", "#9ecae9"), width=.8)
ax.axhline(1, color="black", lw=1); ax.set(title="월별 할인 시작 횟수 (관찰 일수 대비, 1 = 평균)", xlabel="월", ylabel="배율")
plt.xticks(rotation=0); plt.show()
print("평균보다 15% 이상 많은 달:", [int(m) for m, r in ratio.items() if r > 1.15])

### 차트 해설 — H2
막대가 1(검은 선)보다 높은 달이 할인이 평소보다 자주 시작되는 달이다. 빨간 막대가 스토어의 정기 세일 시즌과 겹치는지 본다.
**주의**: 같은 세일에 여러 게임이 한꺼번에 들어가므로 게임들은 서로 독립이 아니다. p값은 실제보다 작게 나오기 쉬워서, 효과 크기(Cramér's V)를 함께 본다.

### H3. 서비스의 "평균 간격" 규칙은 얼마나 정확한가?
**백테스트**: 각 게임에서 k번째 할인 시작일을 앞의 k-1회만 보고 예측해, 실제와의 오차(일)를 잰다.

| 방법 | 예측 |
|---|---|
| **서비스 규칙** | 직전 할인일 + 과거 간격의 평균 |
| 기준 A (직전 간격) | 직전 할인일 + 직전 간격 하나 |
| 기준 B (전체 중앙값) | 직전 할인일 + 모든 게임 간격의 중앙값 (게임 정보를 쓰지 않는 가장 단순한 값) |

In [ ]:
global_gap = int(np.median(pooled))
bt = []
for gid, ev in events.items():
    st = ev.start.tolist()
    for k in range(2, len(st)):               # 앞의 k회(최소 2회)로 k번째를 예측
        past = st[:k]
        gaps = [(b - a).days for a, b in zip(past, past[1:])]
        actual = st[k]
        bt.append({
            "game_id": gid, "actual": actual,
            "err_rule": abs((past[-1] + pd.Timedelta(days=round(np.mean(gaps))) - actual).days),
            "err_last": abs((past[-1] + pd.Timedelta(days=gaps[-1]) - actual).days),
            "err_global": abs((past[-1] + pd.Timedelta(days=global_gap) - actual).days),
        })
bt = pd.DataFrame(bt)
print(f"백테스트 예측 {len(bt):,}건 ({bt.game_id.nunique():,}개 게임) · 전체 중앙값 간격 {global_gap}일")

res = pd.DataFrame({
    "MAE(일)": bt[["err_rule", "err_last", "err_global"]].mean(),
    "중앙 오차(일)": bt[["err_rule", "err_last", "err_global"]].median(),
    "±14일 적중률": (bt[["err_rule", "err_last", "err_global"]] <= 14).mean(),
}).round(2)
res.index = ["서비스 규칙(평균 간격)", "기준 A(직전 간격)", "기준 B(전체 중앙값)"]
display(res)

w3a = stats.wilcoxon(bt.err_rule, bt.err_last, alternative="less")
w3b = stats.wilcoxon(bt.err_rule, bt.err_global, alternative="less")
print(f"규칙 < 기준 A  : p = {w3a.pvalue:.4g}")
print(f"규칙 < 기준 B  : p = {w3b.pvalue:.4g}")

bt[["err_rule", "err_last", "err_global"]].plot.box(figsize=(7, 3.8), showfliers=False)
plt.xticks([1, 2, 3], ["서비스 규칙", "기준 A", "기준 B"]); plt.ylabel("절대 오차(일)"); plt.title("예측 오차 비교")
plt.show()

### 차트 해설 — H3
상자가 낮을수록 예측이 정확하다. **서비스 규칙이 두 기준보다 낮고, 검정에서도 유의하면** 평균 간격이라는 단순 규칙이 쓸 만하다는 근거다.
셋이 비슷하면 "예측이 사실상 맞히기 어렵다"는 뜻이라 **예측일을 하나로 단정하지 말고 구간으로 보여 주는 쪽**이 맞다.

### H3 확장 — 게임별 규칙이 정말 도움이 되는가?
H3-b에서 서비스 규칙이 "전체 중앙값(모든 게임의 간격 중앙값)"보다 나쁘게 나왔다. 두 가지를 더 확인한다.

1. **평균 대신 중앙값**을 쓰면 나아지는가? (평균은 가끔 있는 긴 간격에 끌려간다)
2. 위의 "전체 중앙값"은 **예측하는 시점보다 미래의 간격까지 포함**해서 계산했다. 그래서 규칙에 불리한 비교일 수 있다.
   예측 시점 **이전에 끝난 간격만** 써서 다시 계산한 값(`as-of`)과도 비교한다.

In [ ]:
# 시점별 전체 중앙값(as-of): 예측 대상 날짜 이전에 끝난 모든 게임의 간격만으로 계산
gap_log = pd.DataFrame(
    [(ev.start.iloc[i], (ev.start.iloc[i] - ev.start.iloc[i - 1]).days)
     for ev in events.values() for i in range(1, len(ev))],
    columns=["end", "gap"]).sort_values("end")
end_arr, gap_arr = gap_log.end.to_numpy(), gap_log.gap.to_numpy()

def global_gap_asof(t, min_n=50):
    n = np.searchsorted(end_arr, np.datetime64(t), side="left")   # t보다 먼저 끝난 간격 수
    return float(np.median(gap_arr[:n])) if n >= min_n else np.nan

rows = []
for gid, ev in events.items():
    st = ev.start.tolist()
    for k in range(2, len(st)):
        past, actual = st[:k], st[k]
        gaps = [(b - a).days for a, b in zip(past, past[1:])]
        err = lambda off: abs((past[-1] + pd.Timedelta(days=float(off)) - actual).days)
        g_asof = global_gap_asof(actual)
        rows.append({"game_id": gid,
                     "서비스 규칙(평균 간격)": err(round(np.mean(gaps))),
                     "중앙값 규칙": err(np.median(gaps)),
                     "전체 중앙값(미래 포함)": err(global_gap),
                     "전체 중앙값(과거만)": err(g_asof) if not np.isnan(g_asof) else np.nan})
bt2 = pd.DataFrame(rows).dropna()      # 과거 간격이 50개 미만인 초기 예측은 제외
cols = [c for c in bt2.columns if c != "game_id"]
print(f"비교 대상 예측 {len(bt2):,}건 ({bt2.game_id.nunique():,}개 게임)")

h3x = pd.DataFrame({
    "MAE(일)": bt2[cols].mean(),
    "중앙 오차(일)": bt2[cols].median(),
    "±7일 적중률": (bt2[cols] <= 7).mean(),
    "±14일 적중률": (bt2[cols] <= 14).mean(),
}).round(2)
display(h3x)

p_med_vs_mean = stats.wilcoxon(bt2["중앙값 규칙"], bt2["서비스 규칙(평균 간격)"], alternative="less").pvalue
p_med_vs_global = stats.wilcoxon(bt2["중앙값 규칙"], bt2["전체 중앙값(과거만)"], alternative="less").pvalue
p_mean_vs_global = stats.wilcoxon(bt2["서비스 규칙(평균 간격)"], bt2["전체 중앙값(과거만)"], alternative="less").pvalue
print(f"중앙값 규칙 < 평균 규칙           : p = {p_med_vs_mean:.4g}")
print(f"중앙값 규칙 < 전체 중앙값(과거만)  : p = {p_med_vs_global:.4g}")
print(f"평균 규칙   < 전체 중앙값(과거만)  : p = {p_mean_vs_global:.4g}")

In [ ]:
# 오차 허용 범위별 적중률과, "예측일 ± N일" 구간을 얼마나 넓혀야 하는지
tol = np.arange(0, 61)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for c in cols:
    ax[0].plot(tol, [(bt2[c] <= t).mean() for t in tol], label=c)
ax[0].set(title="오차 허용 범위별 적중률", xlabel="허용 오차(일)", ylabel="실제 할인이 그 안에 시작된 비율")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

q = bt2[cols].quantile([.5, .8, .9]).T
q.columns = ["50%", "80%", "90%"]
q.plot.barh(ax=ax[1]); ax[1].set(title="예측일 ± N일 구간의 폭 (N)", xlabel="일")
plt.tight_layout(); plt.show()
print("예측일 ± N일 안에 실제 할인이 시작될 확률 (N = 오차의 분위수)")
q.round(0).astype(int)

### 차트 해설 — H3 확장
- **왼쪽**: 곡선이 위에 있을수록 같은 허용 오차에서 더 자주 맞힌다. 곡선들이 거의 겹치면 어느 규칙을 써도 정확도가 같다는 뜻이다.
- **오른쪽**: "예측일 ± N일"로 보여 줄 때 실제 할인이 그 구간 안에 들어올 확률(50/80/90%)에 필요한 N이다. 날짜 하나를 단정하기보다 **구간으로 안내**할 때의 근거가 된다.

### (선택) 계절성 확인 — Prophet
H2를 다른 방법으로 확인한다. 할인율 시계열 하나에 Prophet을 맞춰 **연간 계절 성분**을 본다. 이력이 2년이 안 돼서 예측용이 아니라 **계절 성분을 시각화하는 용도**로만 쓴다.
설치가 안 된 PC에서는 이 칸을 건너뛴다.

In [ ]:
try:
    from prophet import Prophet

    # 이력이 가장 긴 게임 하나를 골라 일별 할인율 시계열로 만든다 (스토어 최저가 기준)
    gid = max(events, key=lambda g: len(events[g]))
    d = series_by_game[gid].set_index("day").discount_pct
    cal = d[~d.index.duplicated(keep="last")].reindex(pd.date_range(d.index.min(), END)).ffill()
    df_p = pd.DataFrame({"ds": cal.index, "y": cal.values})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(df_p)
    fc = m.predict(m.make_future_dataframe(periods=120))
    m.plot_components(fc); plt.show()
    print("대상 게임:", games.set_index("game_id").name_ko.get(gid, gid))
except ImportError:
    print("prophet 미설치 — 건너뜀 (pip install prophet)")

## 5단계. 결론 도출

In [ ]:
def verdict(p, alpha=0.05):
    return "채택 (유의)" if p < alpha else "기각 못 함 (유의하지 않음)"

result = pd.DataFrame([
    ["H1 할인 간격이 무작위보다 규칙적", f"관측 CV 중앙값 {gap_df.cv.median():.2f} vs 무작위 {gap_df.cv_null.median():.2f}", w1.pvalue, verdict(w1.pvalue)],
    ["H2 할인이 특정 달에 몰림", f"Cramér's V = {cramers_v:.2f}", p_month, verdict(p_month)],
    ["H3-a 서비스 규칙이 직전 간격보다 정확", f"MAE {bt.err_rule.mean():.1f}일 vs {bt.err_last.mean():.1f}일", w3a.pvalue, verdict(w3a.pvalue)],
    ["H3-b 서비스 규칙이 전체 중앙값보다 정확", f"MAE {bt.err_rule.mean():.1f}일 vs {bt.err_global.mean():.1f}일", w3b.pvalue, verdict(w3b.pvalue)],
    ["H3-c 중앙값 규칙이 평균 규칙보다 정확", f"MAE {bt2['중앙값 규칙'].mean():.1f}일 vs {bt2['서비스 규칙(평균 간격)'].mean():.1f}일", p_med_vs_mean, verdict(p_med_vs_mean)],
    ["H3-d 중앙값 규칙이 전체 중앙값(과거만)보다 정확", f"MAE {bt2['중앙값 규칙'].mean():.1f}일 vs {bt2['전체 중앙값(과거만)'].mean():.1f}일", p_med_vs_global, verdict(p_med_vs_global)],
], columns=["가설", "핵심 수치", "p값", "판정(α=0.05)"])
result["p값"] = result["p값"].map(lambda p: f"{p:.4g}")
result

### 해석과 인사이트
> 2026-11-26 실행 기준 · 데이터 2024-12-07 ~ 2026-11-26, 1,000개 게임 (할인이 4회 이상인 683개 게임으로 간격 분석)

| 가설 | 결과 | 의미 |
|---|---|---|
| H1 규칙성 | **채택** · 변동계수 중앙값 0.54 (무작위 기준 0.94), 85.5%의 게임이 무작위보다 규칙적 | 할인에는 어느 정도 리듬이 있다 |
| H2 시즌성 | **채택, 효과는 작음** · 3·6·12월이 평균보다 15% 이상 많음, Cramér's V = 0.08 | 시즌 영향은 있지만 약하다. p값이 극히 작은 것은 표본이 크고 게임들이 같은 세일에 묶여 있기 때문 |
| H3 예측 정확도 | **부분 채택** · 직전 간격 하나보다는 정확(MAE 23.2일 vs 27.4일). 그러나 **게임 구분 없는 전체 중앙값(약 35일)보다 나쁨**(MAE 20.7일) | 게임별로 평균 간격을 구하는 수고가 정확도로 이어지지 않았다 |

**H3에서 확인한 것**
- 평균 대신 **중앙값**을 써도 차이가 없다 (MAE 22.6일, 평균 규칙과 p=0.13).
- 전체 중앙값이 유리했던 이유가 "미래 데이터를 써서"는 아니다. 예측 시점 **이전 데이터만**으로 계산해도 결과가 같았다 (MAE 20.7일, 규칙 대비 p≈0.9998).
- 어느 규칙이든 ±14일 안에 맞는 비율은 약 50%다. 실제 할인은 예측일 ±14일(50%), ±24~29일(80%), ±41~45일(90%) 범위에 들어온다.

### 서비스에 주는 제안
1. **"다음 할인 예상일"을 날짜 하나가 아니라 구간으로 보여 준다.** 예) "○○ 전후 (±14일 50%, ±4주 80%)". 날짜 하나로 단정하기에는 절반만 2주 안에 맞는다.
2. **게임별 평균 간격에 의존하지 말고, 이력이 짧은 게임은 전체 중앙값(약 35일)을 기본값으로 쓴다.** 현재 규칙(평균)이 전체 중앙값보다 나을 이유를 이번 데이터에서 찾지 못했다.
3. **3·6·12월 세일 시즌 안내는 보조 정보로만 쓴다.** 효과가 작아서 구매 판단의 핵심 근거가 되기는 어렵다.
4. 규칙 변경은 `docs/contracts/timing_rules.md`에 적고 backend 담당(김준영)에게 전달한다. 이 노트북은 근거 자료다.

### 한계
- "할인"은 **모든 스토어 중 하나라도 10% 이상이면 1회**로 센다. 키 판매점이 많이 섞여 있어 간격이 짧게(중앙값 약 34일) 나왔다. 공식 스토어 기준이나 더 큰 할인(예: 30% 이상)만 보면 결과가 다를 수 있다 (**미검증**).
- 이력이 719일이라 연간 패턴은 최대 2회분이다. Prophet은 "730일 미만" 경고가 떠서 보조 시각화로만 썼다.
- 분석 대상은 한국어 리뷰가 많은 인기작 위주(`analysis/game_list/README.md`)라 전체 게임으로 일반화할 수 없다.
- 같은 세일에 많은 게임이 같이 들어가므로 게임 간 독립을 가정한 p값은 실제보다 작게 나온다.